In [ ]:
# ============================================
# 1) Instalación de dependencias (Colab)
# ============================================
# Nota: Reinicia el colab si es la primera vez que instalas (Runtime > Restart session) para limpiar conflictos.
!pip -q install langgraph==0.2.39 langchain==0.3.7 pydantic==2.9.2 typing_extensions==4.12.2

# ============================================
# 2) Importaciones y utilidades
# ============================================
from __future__ import annotations
from typing import TypedDict, Literal, List, Dict, Any, Optional
from dataclasses import dataclass
from pprint import pprint
from textwrap import dedent # elimina la indentación innecesaria al inicio de cada línea de un texto multilínea.

# LangGraph
from langgraph.graph import StateGraph, END

# ============================================
# 3) Contexto teórico (en comentarios)
# --------------------------------------------
# - LangGraph modela el flujo como un GRAFO DE ESTADOS: cada nodo transforma el estado,
#   y el enrutamiento entre nodos puede ser condicional y cíclico (máquina de estados).
# - HITL (Human-in-the-Loop): el grafo "pausa" en un punto crítico y espera input humano.
# - En este ejemplo: GENERACIÓN -> HITL -> RUTEADOR -> (APROBADO -> FINAL) / (CAMBIOS/RECHAZO -> GENERACIÓN)
# - Se demuestra persistencia de estado, uso de tools y redirección dinámica por veredicto humano.

# ============================================
# 4) Definición de "tools" simuladas Empresa Tecnología
# --------------------------------------------
#   Son funciones simples que devuelven datos ficticios y realizan cálculos/validaciones. 3
#   Las llamamos desde los nodos del grafo.
# ============================================

@dataclass
class Tool:
    name: str
    description: str

    def __call__(self, *args, **kwargs):
        raise NotImplementedError
# 1 de 3
class GetBudgetDataTool(Tool):
    def __init__(self):
        super().__init__(
            name="get_budget_data",
            description="Obtiene items de costo ficticios (descripcion, cantidad, precio_unitario)."
        )

    def __call__(self) -> List[Dict[str, Any]]:
        # Datos de ejemplo para un reporte de costos simple
        return [
            {"descripcion": "Servicios de nube", "cantidad": 3, "precio": 120.0},
            {"descripcion": "Horas de consultoría", "cantidad": 12, "precio": 45.5},
            {"descripcion": "Licencias software", "cantidad": 2, "precio": 230.0},
        ]
#2 de 3
class CalcTotalsTool(Tool):
    def __init__(self):
        super().__init__(
            name="calculate_totals",
            description="Calcula subtotal, impuestos y total."
        )

    def __call__(self, items: List[Dict[str, Any]], tax_rate: float = 0.19) -> Dict[str, float]:
        subtotal = sum(i["cantidad"] * i["precio"] for i in items)
        tax = round(subtotal * tax_rate, 2)
        total = round(subtotal + tax, 2)
        return {"subtotal": round(subtotal, 2), "impuestos": tax, "total": total}

# 3 de 3
class ValidateCurrencyTool(Tool):
    def __init__(self):
        super().__init__(
            name="validate_currency",
            description="Valida si la moneda es soportada."
        )

    def __call__(self, currency_code: str, allowed=("USD","COP","EUR")) -> bool:
        return currency_code.upper() in allowed

get_budget_data = GetBudgetDataTool()
calculate_totals = CalcTotalsTool()
validate_currency = ValidateCurrencyTool()

# ============================================
# 5) Definición del ESTADO del grafo
# --------------------------------------------
#   El estado persiste y se enriquece en cada nodo.
# ============================================

class ReportState(TypedDict, total=False):
    # Datos base / tools
    items: List[Dict[str, Any]]
    currency: str
    totals: Dict[str, float]

    # Iteraciones / control
    iteration: int
    draft_report: str
    supervisor_verdict: Literal["aprobado","rechazado","solicitar cambios"] | None
    supervisor_notes: Optional[str]

    # Historial para trazabilidad educativa
    history: List[str]

def pretty_state(state: ReportState, title: str = "ESTADO ACTUAL"):
    print("\n" + "="*80)
    print(f"🔎 {title}")
    print("="*80)
    pprint(dict(state))
    print("="*80 + "\n")

# ============================================
# 6) NODOS DEL GRAFO (máquina de estados)
# --------------------------------------------
#   Cada nodo es una función que recibe y retorna parches de estado (dicts parciales).
#   LangGraph combina esos parches sobre el estado persistente.
# ============================================

def node_collect_data(state: ReportState) -> ReportState:
    """
    Nodo: 'collect'
    - Usa tools para obtener items, validar moneda y calcular totales.
    - En una app real, aquí podrían invocarse APIs/DBs o herramientas LangChain Tools.
    """
    history = state.get("history", [])
    history.append("collect: recopilando datos con tools")

    # 1) Items (tool)
    items = get_budget_data()
    # 2) Moneda (tool)
    currency = state.get("currency", "USD")
    if not validate_currency(currency):
        # Si no es válida, forzamos USD para continuar el ejemplo educativo
        history.append(f"collect: moneda '{currency}' no válida; usando 'USD'")
        currency = "USD"
    # 3) Totales (tool)
    totals = calculate_totals(items)

    out = {
        "items": items,
        "currency": currency,
        "totals": totals,
        "history": history
    }
    pretty_state(out, "Salida de nodo 'collect'")
    return out

def render_report(items: List[Dict[str, Any]], totals: Dict[str, float], currency: str, notes: Optional[str], iteration: int) -> str:
    lines = [
        f"# REPORTE DE COSTOS (iteración {iteration})",
        "",
        "## Ítems",
        "| Descripción              | Cantidad | Precio |",
        "|--------------------------|----------|--------|",
    ]
    for i in items:
        lines.append(f"| {i['descripcion']:<24} | {i['cantidad']:^8} | {i['precio']:^6.2f} |")
    lines.extend([
        "",
        "## Resumen",
        f"- Subtotal: {totals['subtotal']:.2f} {currency}",
        f"- Impuestos: {totals['impuestos']:.2f} {currency}",
        f"- TOTAL: **{totals['total']:.2f} {currency}**",
        "",
    ])
    if notes:
        lines.extend([
            "## Notas del supervisor (previas)",
            f"> {notes}",
            ""
        ])
    lines.append("_Borrador generado automáticamente para revisión humana._")
    return "\n".join(lines)

def node_generate(state: ReportState) -> ReportState:
    """
    Nodo: 'generate'
    - Genera un borrador de reporte usando el estado (items, totals, notes previas).
    - Si hubo notas del supervisor, se incorporan como contexto.
    """
    history = state.get("history", [])
    iteration = state.get("iteration", 0) + 1
    history.append(f"generate: creando borrador (iteración {iteration})")

    draft = render_report(
        items=state["items"],
        totals=state["totals"],
        currency=state["currency"],
        notes=state.get("supervisor_notes"),
        iteration=iteration
    )

    out = {
        "draft_report": draft,
        "iteration": iteration,
        "history": history
    }
    pretty_state(out, "Salida de nodo 'generate'")
    print("📝 BORRADOR DE REPORTE:\n")
    print(draft)
    print("\n" + "-"*80)
    return out

def node_hitl(state: ReportState) -> ReportState:
    """
    Nodo: 'hitl' (Human-in-the-Loop)
    - Pausa para revisión humana usando input().
    - El usuario (supervisor) ingresa veredicto y notas.
    """
    history = state.get("history", [])
    history.append("hitl: esperando validación humana (input)")

    print("\n👤 ***REVISIÓN HUMANA REQUERIDA***")
    print("Opciones de veredicto: 'aprobado' | 'rechazado' | 'solicitar cambios'")
    while True:
        verdict = input("Ingrese veredicto del supervisor: ").strip().lower()
        if verdict in ("aprobado","rechazado","solicitar cambios"):
            break
        print("⚠️ Veredicto inválido. Intente nuevamente.")

    notes = ""
    if verdict in ("rechazado","solicitar cambios"):
        notes = input("Ingrese notas breves del supervisor (qué cambiar/mejorar): ").strip()

    out = {
        "supervisor_verdict": verdict,
        "supervisor_notes": notes if notes else None,
        "history": history
    }
    pretty_state(out, "Salida de nodo 'hitl'")
    return out

def route_by_verdict(state: ReportState) -> Literal["finalize", "generate"]:
    """
    Función de enrutamiento condicional.
    - Si 'aprobado' -> 'finalize'
    - Si 'rechazado' o 'solicitar cambios' -> 'generate' (ciclo con notas)
    """
    verdict = state.get("supervisor_verdict")
    if verdict == "aprobado":
        return "finalize"
    # Para ambos casos, regresamos al generador con las notas como contexto
    return "generate"

def node_finalize(state: ReportState) -> ReportState:
    """
    Nodo: 'finalize'
    - Cierre del flujo. Limpia banderas y registra histórico.
    """
    history = state.get("history", [])
    history.append("finalize: reporte aprobado y finalizado")
    print("\n✅ ***REPORTE APROBADO***")
    print("Se alcanzó el estado final. Versión aprobada del reporte:\n")
    print(state["draft_report"])
    out = {
        "history": history
    }
    pretty_state(out, "Salida de nodo 'finalize'")
    return out

# ============================================
# 7) CONSTRUCCIÓN DEL GRAFO DE ESTADOS (LangGraph)
# --------------------------------------------
#   Nodos: collect -> generate -> hitl -> (router condicional) -> finalize|generate
#   Ciclos: si 'rechazado' o 'solicitar cambios', se regresa a 'generate'
# ============================================

def build_app():
    graph = StateGraph(ReportState)

    # Registrar nodos
    graph.add_node("collect", node_collect_data)
    graph.add_node("generate", node_generate)
    graph.add_node("hitl", node_hitl)
    graph.add_node("finalize", node_finalize)

    # Definir inicio
    graph.set_entry_point("collect")

    # Transiciones lineales iniciales
    graph.add_edge("collect", "generate")
    graph.add_edge("generate", "hitl")

    # Enrutamiento condicional desde HITL
    graph.add_conditional_edges(
        "hitl",
        route_by_verdict,
        {
            "finalize": "finalize",
            "generate": "generate"
        },
    )

    # Estado final
    graph.add_edge("finalize", END)

    app = graph.compile()
    return app

# ============================================
# 8) Función de ejecución con estado inicial
# --------------------------------------------
#   - Muestra el recorrido y el estado en cada paso.
#   - Puedes correr múltiples veces con distintos veredictos para ver el ciclo.
# ============================================

def run_workflow(currency: str = "USD"):
    print("\n🚀 INICIANDO WORKFLOW LangGraph: Aprobación de Reportes de Costos")
    app = build_app()

    # Estado inicial con la clase ReportState
    init_state: ReportState = {
        "currency": currency,
        "iteration": 0,
        "history": ["start: estado inicial"]
    }
    pretty_state(init_state, "Estado inicial (antes de invocar el grafo)")

    # Invocar el grafo. Se detendrá en 'hitl' hasta que ingreses input en Colab.
    final_state = app.invoke(init_state)

    print("\n🎯 WORKFLOW COMPLETADO")
    pretty_state(final_state, "Estado final luego de recorrer el grafo")
    print("Historial (rastro de ejecución):")
    for i, step in enumerate(final_state.get("history", []), 1):
        print(f"  {i:02d}. {step}")

# ============================================
# 9) DEMO
# --------------------------------------------
# Descomenta la siguiente línea y ejecuta la celda para correr el flujo.
# En la revisión humana, ingresa uno de: aprobado | rechazado | solicitar cambios
# Si eliges 'rechazado' o 'solicitar cambios', añade notas y observa el ciclo.
# ============================================

run_workflow("COP")


🚀 INICIANDO WORKFLOW LangGraph: Aprobación de Reportes de Costos

🔎 Estado inicial (antes de invocar el grafo)
{'currency': 'COP', 'history': ['start: estado inicial'], 'iteration': 0}


🔎 Salida de nodo 'collect'
{'currency': 'COP',
 'history': ['start: estado inicial', 'collect: recopilando datos con tools'],
 'items': [{'cantidad': 3, 'descripcion': 'Servicios de nube', 'precio': 120.0},
           {'cantidad': 12,
            'descripcion': 'Horas de consultoría',
            'precio': 45.5},
           {'cantidad': 2,
            'descripcion': 'Licencias software',
            'precio': 230.0}],
 'totals': {'impuestos': 259.54, 'subtotal': 1366.0, 'total': 1625.54}}


🔎 Salida de nodo 'generate'
{'draft_report': '# REPORTE DE COSTOS (iteración 1)\n'
                 '\n'
                 '## Ítems\n'
                 '| Descripción              | Cantidad | Precio |\n'
                 '|--------------------------|----------|--------|\n'
                 '| Servicios de nube    